| 순위 | 모델 (model) | RMSE (mm) | R² |
| :---: | :--- | :---: | :---: |
| 1 | Linear | 0.004025 | 0.934886 |
| 2 | XGBoost | 0.004675 | 0.912121 |
| 3 | RandomForest | 0.005072 | 0.896720 |

In [17]:
import numpy as np
from joblib import load
import pandas as pd
from sklearn.model_selection import train_test_split

# 목표값, 규격하한, 규격상한
SV, LSL, USL = 10.00, 9.95, 10.05
# 공차의 중간점 지정
half_tol = (USL - LSL) / 2

# 이전 도출한 내용과 모델을 활용한다.
# model : XGBoost | RMSE : 0.004675 | R2:0.912121
rmse = 0.004675
best = load('/content/data/best.joblib')

# 데이터도 호출
X = pd.read_csv('/content/data/X_features.csv')
y = pd.read_csv('/content/data/y_target.csv').squeeze()
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)

In [ ]:
pred = best.predict(x_test) # 제품 치수 예측값

# 예측값, 실제값
risk = pd.DataFrame({'pred': pred, 'actual': y_test.values}, index=x_test.index)

# 예측값이 목표값(SV) 10 에 얼마나 떨어진지를 구해 공차 중앙점과 얼마나 차이가 나는지...
risk['proximity'] = (risk['pred'] - SV).abs() / half_tol # Proximity : 근접

# 예측값에 모델 오차 2배(2 * rmse)를 더했을 때 상한 규격(USL)10.05 를 초과하는지 확인
# 예측값에 모델 오차 2배(2 * rmse)를 더뺐을 때 하한 규격(LSL)9.95 미만인지 확인
# 둘 중하나라도 해당되면 True
risk['out_band']  = (risk['pred'] + 2 * rmse > USL) | (risk['pred'] - 2 * rmse < LSL)

In [19]:
# select([조건리스트],[조건1 대응값,조건2 대응값],기본값)
risk['grade'] = np.select(
    [risk['out_band'] | (risk['proximity'] >= 0.8), risk['proximity'] >= 0.6],
    ['위험', '주의'], default='정상')

print(risk['grade'].value_counts())

grade
정상    573
주의     25
위험      1
Name: count, dtype: int64


In [22]:
# 저장
risk.to_csv('/content/data/risk.csv', index=False)